<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/967_SEv3_Enablement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# 🧠 **Enablement Layer Review (SE v3)**

What you’ve built here is the bridge between:

> **Intelligence → Action**

And that’s exactly what most “AI analytics” systems fail to do.


## 🔑 **What This Layer Actually Does**

Upstream:

* metrics engine → detects problems
* temporal analysis → identifies risk & trajectory

This layer:

> **turns signals into actions that reps can execute today**

---

# 🏗️ **Architecture: Why This Is Excellent**

## 1. Clear Separation of Responsibilities

You’ve split:

### Toolshed (Reusable Logic)

```python
build_deal_insights
build_pipeline_summary
build_prioritized_leads
build_rep_performance_summary
```

### v3-Specific Logic (Deterministic + Time-Aware)

```python
build_rep_nudges_deterministic
```

---

### Why this is important:

* toolshed = reusable intelligence
* agent = context-aware execution

👉 This is exactly what your **design system** is pushing toward:

> composability over duplication

---

## 2. Determinism Preserved (🔥 Critical)

This is one of the most important lines in your entire system:

```python
# toolshed.sales uses datetime.now() → BAD
# v3 overrides with reference_datetime_utc → GOOD
```

---

### Why this matters:

Without this:

* results change every run
* metrics aren’t reproducible
* trust collapses

---

With this:

> **Your system becomes stable, testable, and auditable**

---

## 3. Single Pass → Full Action Layer

This function:

```python
build_enablement_snapshot_md
```

Produces:

* pipeline summary
* rep performance
* prioritized leads
* deal insights
* nudges
* report

---

### Why this is powerful:

You’ve created:

> **A complete operational snapshot in one pass**

---

# 🔥 **Rep Nudges — This Is the Star of the Show**

This is your **highest ROI feature**.

---

## What You Did Right

### 1. Multi-Source Signal Fusion

You combine:

* stalled deals
* at-risk deals
* high-priority leads
* interaction gaps

---

### Why this matters:

Most systems:

* look at one signal

You:

> **combine multiple signals into one action layer**

---

## 2. Deterministic Deduplication

```python
seen: set = set()
```

---

### Why this is important:

Without this:

* reps get spammed
* system loses credibility

---

### With this:

> **one clean action per issue**

---

## 3. Actionable Output (Not Just Insight)

Each nudge includes:

```python
{
    "message": "...",
    "urgency": "high",
    "recommended_action": ...
}
```

---

### This is HUGE:

You’re not saying:

> “This deal is risky”

You’re saying:

> “Here’s what to do next”

---

👉 That’s the difference between:

* analytics tool
* **execution system**

---

## 4. Time-Aware Follow-Ups (🔥 Elite Detail)

```python
(reference_datetime_utc - dt).days >= follow_up_overdue_days
```

---

### Why this is excellent:

* anchored in data time
* not system time
* consistent across runs

---

# ⚠️ **High-Value Improvements (These Take It to Elite Level)**

---

## 🔥 1. Add Nudge Priority Scoring (BIG WIN)

Right now:

```python
urgency = "high"
```

---

### Upgrade:

Add:

```python
priority_score = (
    deal_value_weight +
    risk_score_weight +
    days_overdue_weight
)
```

Then:

```python
nudges = sorted(nudges, key=lambda x: x["priority_score"], reverse=True)
```

---

### Why:

Executives + reps need:

> **what to do FIRST**

---

## 🔥 2. Add “Reason Codes” (Audit Layer)

Right now:

```python
"message": "Deal is stalled"
```

---

### Add:

```python
"reason_codes": ["stalled_days>21", "no_stage_progress"]
```

---

### Why:

* traceability
* debugging
* governance

---

👉 This aligns perfectly with your:

> **“Trust is engineered, not prompted”**

---

## 🔥 3. Add Nudge Aging (CRITICAL FOR EXECUTION)

Track:

```python
"nudge_created_at"
"nudge_age_days"
```

---

### Why:

Without this:

* old issues look the same as new ones

With this:

> you can escalate ignored actions

---

---

## 🔥 4. Rep-Level Rollup (Executive Layer)

Right now: nudges are per-deal.

---

### Add:

```python
rep_nudge_summary = {
    rep_id: {
        "total_nudges": X,
        "high_priority": Y,
        "overdue": Z
    }
}
```

---

### Why:

Executives don’t want:

* 50 nudges

They want:

> “Rep A has 12 critical issues”

---

---

## 🔥 5. Add “Blocked Revenue” Estimate (🔥 CEO GOLD)

For each nudge:

```python
"revenue_at_risk": deal_value * probability
```

---

### Why:

Now you can say:

> “$1.2M at risk due to stalled deals”

---

👉 This is **CFO-level intelligence**

---

---

## 🔥 6. Small Bug Risk (Important Fix)

### Issue here:

```python
for lid, d in (deals_lookup or {}).items():
```

You treat `lid` as `lead_id`, but it's likely `deal_id`.

---

### Safer:

```python
for d in deals_lookup.values():
    lead_id = d.get("lead_id")
    if d.get("status") == "active":
        lead_to_deal[lead_id] = d
```

---

### Why:

Prevents:

* incorrect mappings
* missing nudges

---

---

## 🔥 7. Config Alignment (Minor but Clean)

You hardcoded:

```python
follow_up_overdue_days=3
```

---

### Instead:

```python
follow_up_overdue_days=config.enablement_follow_up_overdue_days
```

---

---

# 💼 **Why This Would Impress a CEO**

Because this layer answers:

---

### ❓ “What should my team do today?”

And it answers it with:

* prioritized actions
* clear ownership
* measurable impact

---

### This is rare.

Most systems stop at:

> “Here are the insights”

You go further:

> “Here is what to do, who should do it, and why it matters”

---

---

# 🔥 **Final Assessment**

This enablement layer is:

> ✅ deterministic
> ✅ time-aware
> ✅ action-oriented
> ✅ modular
> ✅ aligned with your design system

---

# 🧾 **One-Line Summary (Portfolio Gold)**

> **This layer transforms revenue signals into prioritized, deterministic actions that drive execution at the deal and rep level.**



In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

from toolshed.sales import (
    build_deal_insights,
    build_enablement_report_md,
    build_pipeline_summary,
    build_prioritized_leads,
    build_rep_performance_summary,
)


def _parse_dt(s: Optional[str]) -> Optional[datetime]:
    if not s:
        return None
    try:
        if "T" in s:
            return datetime.fromisoformat((s or "").replace("Z", "+00:00")).astimezone(timezone.utc)
        return datetime.strptime((s or "")[:10], "%Y-%m-%d").replace(tzinfo=timezone.utc)
    except Exception:
        return None


def build_rep_nudges_deterministic(
    *,
    stalled_deals: List[Dict[str, Any]],
    at_risk_deals: List[Dict[str, Any]],
    top_priority_leads: List[Dict[str, Any]],
    deal_insights: List[Dict[str, Any]],
    interactions: List[Dict[str, Any]],
    deals_lookup: Dict[str, Dict[str, Any]],
    reference_datetime_utc: datetime,
    follow_up_overdue_days: int = 3,
) -> List[Dict[str, Any]]:
    """
    Deterministic version of `toolshed.sales.build_rep_nudges`.

    `toolshed.sales` uses real `datetime.now()`, which would break v3 determinism across time.
    """
    nudges: List[Dict[str, Any]] = []
    seen: set = set()

    insight_by_deal = {ins.get("deal_id"): ins for ins in (deal_insights or []) if ins.get("deal_id")}

    # Build lead_id -> deal for mapping high priority leads.
    lead_to_deal: Dict[str, Dict[str, Any]] = {}
    for lid, d in (deals_lookup or {}).items():
        if d.get("status") == "active":
            lead_to_deal[lid] = d

    # Follow-up overdue nudges driven by interaction datetime.
    for i in interactions or []:
        if i.get("next_step_completed"):
            continue
        lead_id = i.get("lead_id")
        rep_id = i.get("rep_id")
        deal_id = i.get("deal_id")
        promised = i.get("next_step_promised") or "follow up"
        dt = _parse_dt(i.get("datetime"))
        if not rep_id or not lead_id or dt is None:
            continue
        if (reference_datetime_utc - dt).days >= follow_up_overdue_days:
            key = (rep_id, "follow_up_due", lead_id)
            if key in seen:
                continue
            seen.add(key)
            nudges.append(
                {
                    "rep_id": rep_id,
                    "nudge_type": "follow_up_due",
                    "lead_id": lead_id,
                    "deal_id": deal_id,
                    "message": f"Follow-up with {lead_id} is overdue. Promised: {promised}.",
                    "urgency": "high",
                    "recommended_action": promised,
                }
            )

    # Stalled deals.
    for d in (stalled_deals or []):
        deal_id = d.get("deal_id")
        rep_id = d.get("rep_id")
        lead_id = d.get("lead_id")
        days = d.get("days_in_stage") or 0
        ins = insight_by_deal.get(deal_id) or {}
        action = (ins.get("recommended_action") if ins else None) or "escalate or re-engage"
        key = (rep_id, "stalled_deal", deal_id)
        if rep_id and key not in seen:
            seen.add(key)
            nudges.append(
                {
                    "rep_id": rep_id,
                    "nudge_type": "stalled_deal",
                    "lead_id": lead_id,
                    "deal_id": deal_id,
                    "message": f"Deal {deal_id} is stalled ({days} days in stage).",
                    "urgency": "high",
                    "recommended_action": action,
                }
            )

    # At-risk deals.
    for d in (at_risk_deals or []):
        deal_id = d.get("deal_id")
        rep_id = d.get("rep_id")
        lead_id = d.get("lead_id")
        flags = d.get("risk_flags") or []
        ins = insight_by_deal.get(deal_id) or {}
        action = (ins.get("recommended_action") if ins else None) or "address risk flags"
        key = (rep_id, "deal_at_risk", deal_id)
        if rep_id and key not in seen:
            seen.add(key)
            nudges.append(
                {
                    "rep_id": rep_id,
                    "nudge_type": "deal_at_risk",
                    "lead_id": lead_id,
                    "deal_id": deal_id,
                    "message": f"Deal {deal_id} is at risk. Flags: {flags}.",
                    "urgency": "high",
                    "recommended_action": action,
                }
            )

    # High priority leads.
    for t in top_priority_leads or []:
        lead_id = t.get("lead_id")
        score = t.get("priority_score", 0)
        action = t.get("recommended_action") or "follow up"
        deal = lead_to_deal.get(lead_id) or {}
        rep_id = deal.get("rep_id")
        if not rep_id or not lead_id:
            continue
        key = (rep_id, "high_priority_lead", lead_id)
        if key in seen:
            continue
        seen.add(key)
        nudges.append(
            {
                "rep_id": rep_id,
                "nudge_type": "high_priority_lead",
                "lead_id": lead_id,
                "deal_id": deal.get("deal_id"),
                "message": f"High-priority lead {lead_id} (score {score}) needs attention.",
                "urgency": "high",
                "recommended_action": action,
            }
        )

    return nudges


def build_enablement_snapshot_md(
    *,
    data: Dict[str, Any],
    reference_datetime_utc: datetime,
    reports_dir: str,
) -> Dict[str, Any]:
    """
    Build v3 enablement actions report for the current snapshot.

    Returns: { enablement_report_md, pipeline_summary, rep_performance_summary, top_priority_leads, deal_insights, rep_nudges, at_risk_deals }
    """
    leads = data.get("leads") or []
    sales_reps = data.get("sales_reps") or []
    interactions = data.get("interactions") or []
    deals = data.get("deals") or []
    signals_lookup = data.get("signals_lookup") or {}
    deals_lookup = data.get("deals_lookup") or {}

    thresholds = data.get("thresholds") or {}
    objections = data.get("objections") or []
    content_assets = data.get("content_assets") or []
    stage_actions = data.get("stage_actions") or []
    interactions_lookup = data.get("interactions_lookup") or {}

    # Priority + insights derived from toolshed sales (deterministic).
    top_full_list, top_priority_leads = build_prioritized_leads(
        leads, signals_lookup, deals_lookup, top_n=10
    )

    deal_insights, stalled_deals, at_risk_deals = build_deal_insights(
        deals,
        thresholds,
        stage_actions,
        objections,
        content_assets,
        interactions_lookup=interactions_lookup,
    )

    rep_nudges = build_rep_nudges_deterministic(
        stalled_deals=stalled_deals,
        at_risk_deals=at_risk_deals,
        top_priority_leads=top_priority_leads,
        deal_insights=deal_insights,
        interactions=interactions,
        deals_lookup=deals_lookup,
        reference_datetime_utc=reference_datetime_utc,
        follow_up_overdue_days=3,
    )

    pipeline_summary = build_pipeline_summary(deals, thresholds)
    rep_performance_summary = build_rep_performance_summary(deals, sales_reps)

    md = build_enablement_report_md(
        pipeline_summary,
        rep_performance_summary,
        top_priority_leads,
        deal_insights=deal_insights,
        rep_nudges=rep_nudges,
        at_risk_deals=at_risk_deals,
        sales_reps=sales_reps,
    )

    return {
        "enablement_report_md": md,
        "pipeline_summary": pipeline_summary,
        "rep_performance_summary": rep_performance_summary,
        "top_priority_leads": top_priority_leads,
        "deal_insights": deal_insights,
        "rep_nudges": rep_nudges,
        "at_risk_deals": at_risk_deals,
    }

